# Setup connection to remote host for MLflow and MinIO UI

In [1]:
group = 0  # Replace with your actual group name or identifier

In [2]:
remote_instances = ['195.148.21.16', '195.148.23.141'] # IP addresses of remote instances
remote_instance = remote_instances[group % len(remote_instances)]

In [3]:
#TODO: shall we add a installation script for ansible or make sure the ansible is installed for all users already on the remote instance?

In [ ]:
# test connections to services
import requests

def test_connection():
    for url in [
        "kserve-gateway.local",
        "ml-pipeline-ui.local",
        "mlflow-server.local",
        "mlflow-minio-ui.local",
        "mlflow-minio.local",
        "prometheus-server.local",
        "grafana-server.local",
        "evidently-monitor-ui.local",
    ]:
        try:
            requests.get(f"http://{url}")
        except Exception as e:
            print(f"Failed to connect to {url}: {e}")
            raise e

test_connection()

Run the following cells in this notebook to port-forward the services.

Access the services from your local browser:
    MLflow: http://localhost:5000
    MinIO UI: http://localhost:9001


In [ ]:
import subprocess
import sys
import time
import signal

def get_pod_name(namespace: str, label_selector: str) -> str:
    """Get the first pod name matching the label in the given namespace."""
    try:
        cmd = [
            "kubectl", "get", "pods", "-n", namespace,
            "-l", label_selector,
            "-o", "jsonpath={.items[0].metadata.name}"
        ]
        pod_name = subprocess.check_output(cmd, text=True).strip()
        return pod_name
    except subprocess.CalledProcessError:
        return None

def start_port_forward(namespace: str, pod_name: str, local_port: int, remote_port: int):
    """Start kubectl port-forward and return the process."""
    cmd = [
        "kubectl", "port-forward", "-n", namespace,
        pod_name, f"{local_port}:{remote_port}"
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(f"Port-forward started: localhost:{local_port} -> {pod_name}:{remote_port}")
    return process

def port_forward_services():
    """
    Forward MLflow and MinIO pods in mlflow namespace to local ports.
    Prints local URLs for access.
    """
    namespace = "mlflow"
    services = {
        "mlflow": 5000,      # local port 5000 -> pod port 5000
        "mlflow-minio": 9001 # local port 9001 -> pod port 9001
    }

    processes = []

    for service_name, local_port in services.items():
        label = f"app={service_name}"
        pod_name = get_pod_name(namespace, label)
        if not pod_name:
            print(f"No pod found for service '{service_name}' in namespace '{namespace}'")
            continue

        process = start_port_forward(namespace, pod_name, local_port, local_port)
        processes.append(process)
        print(f"Access {service_name} at http://localhost:{local_port}")

    return processes

if __name__ == "__main__":
    processes = port_forward_services()
    try:
        while any(p.poll() is None for p in processes):
            time.sleep(1)
    except KeyboardInterrupt:
        print("Stopping all port-forwards...")
        for p in processes:
            if p.poll() is None:
                p.send_signal(signal.SIGINT)
        print("Port-forwards stopped.")


Port-forward started: localhost:5000 -> mlflow-5785498687-7k49w:5000
Access mlflow at http://localhost:5000
Port-forward started: localhost:9001 -> mlflow-minio-844b59c649-t8gl9:9001
Access mlflow-minio at http://localhost:9001
